# Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

# Load Libraries

In [3]:
library(readxl)
library(ggplot2)
library(viridis)
library(dplyr)
library(glue)
library(tidyr)

# Load Data

In [4]:
# List all DEG files
files <- list.files(path = "06_DEG_GO", pattern = "\\.xlsx$", full.names = TRUE)

# View
print(files)

 [1] "06_DEG_GO/MECP2_1155del32_CtO_D111_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx"
 [2] "06_DEG_GO/MECP2_1155del32_CtO_D111_vs_MECP2_R270X_pD_CtO_D90_results_enrichr_results.xlsx"
 [3] "06_DEG_GO/MECP2_1155del32_CtO_D111_vs_X80230F_WT_CtO_D11_results_enrichr_results.xlsx"    
 [4] "06_DEG_GO/MECP2_R270X_CtO_D18_vs_MECP2_1155del32_CtO_D111_results_enrichr_results.xlsx"   
 [5] "06_DEG_GO/MECP2_R270X_CtO_D18_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx"     
 [6] "06_DEG_GO/MECP2_R270X_CtO_D18_vs_MECP2_R270X_pD_CtO_D90_results_enrichr_results.xlsx"     
 [7] "06_DEG_GO/MECP2_R270X_CtO_D18_vs_X80230F_WT_CtO_D11_results_enrichr_results.xlsx"         
 [8] "06_DEG_GO/MECP2_R270X_CtO_D46_vs_MECP2_1155del32_CtO_D111_results_enrichr_results.xlsx"   
 [9] "06_DEG_GO/MECP2_R270X_CtO_D46_vs_MECP2_R270X_CtO_D18_results_enrichr_results.xlsx"        
[10] "06_DEG_GO/MECP2_R270X_CtO_D46_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx"     
[11] "06_DEG_GO/MECP2_R270X_Ct

# Combined Dot Plot

In [5]:
# List of GO databases
databases <- c("GO_Biological_Process_2025", "GO_Cellular_Component_2025", "GO_Molecular_Function_2025",
               "KEGG_2021_Human", "Reactome_2022")

In [6]:
# Create abbreviation mapping
database_abbreviations <- c(
  "GO_Biological_Process_2025" = "GO BP",
  "GO_Cellular_Component_2025" = "GO CC", 
  "GO_Molecular_Function_2025" = "GO MF",
  "KEGG_2021_Human" = "KEGG",
  "Reactome_2022" = "REACTOME"
)

In [22]:
# Initialize an empty dataframe for all GO data
all_data <- data.frame()

In [23]:
# Iterate over each module and read in the corresponding Excel file
for (file in files) {
  for (database in databases) {
    try({
      # Read the data from the Excel file
      df <- read_excel(file, sheet = database)
      
      if (nrow(df) == 0) {
        next  # Skip to the next database if no data
      }
      
      # Select the required columns and add module and database information
      df <- df %>%
        filter(Adjusted.P.value <= 0.05) %>%
        select(Term, Adjusted.P.value, Odds.Ratio) %>%
        mutate(Module = file, Database = database) %>%
        arrange(Adjusted.P.value) %>%  
        slice_head(n = 2) 
      
      # Append to the dataframe
      all_data <- bind_rows(all_data, df)
      
    }, silent = TRUE)
  }
}

# Check if any data was read
if (nrow(all_data) == 0) {
  stop("No data read from any of the Excel files.")
}

# Display the combined dataframe
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Macromolecule Biosynthetic Process (GO:0009059),1.507140e-04,4.766026,06_DEG_GO/MECP2_1155del32_CtO_D111_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx,GO_Biological_Process_2025
2,Cytoplasmic Translation (GO:0002181),2.668454e-04,6.427842,06_DEG_GO/MECP2_1155del32_CtO_D111_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx,GO_Biological_Process_2025
3,Lytic Vacuole Membrane (GO:0098852),4.291520e-07,4.151190,06_DEG_GO/MECP2_1155del32_CtO_D111_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx,GO_Cellular_Component_2025
4,Lysosomal Membrane (GO:0005765),1.391321e-06,3.595585,06_DEG_GO/MECP2_1155del32_CtO_D111_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx,GO_Cellular_Component_2025
5,Cadherin Binding (GO:0045296),1.528400e-05,3.781220,06_DEG_GO/MECP2_1155del32_CtO_D111_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx,GO_Molecular_Function_2025
6,Ubiquitin-Like Protein Ligase Binding (GO:0044389),2.132971e-02,2.867179,06_DEG_GO/MECP2_1155del32_CtO_D111_vs_MECP2_R270X_pA_CtO_D90_results_enrichr_results.xlsx,GO_Molecular_Function_2025


In [24]:
# Format comparison name
all_data$Module <- gsub("06_DEG_GO/", "", all_data$Module)
all_data$Module <- gsub("_results_enrichr_results.xlsx", "", all_data$Module)
all_data$Module <- gsub("_CtO_", " ", all_data$Module)
all_data$Module <- gsub("_", " ", all_data$Module)
all_data$Module <- gsub("MECP2", "", all_data$Module)

# View
head(all_data)

,Term,Adjusted.P.value,Odds.Ratio,Module,Database
,<chr>,<dbl>,<dbl>,<chr>,<chr>
1,Macromolecule Biosynthetic Process (GO:0009059),1.507140e-04,4.766026,1155del32 D111 vs R270X pA D90,GO_Biological_Process_2025
2,Cytoplasmic Translation (GO:0002181),2.668454e-04,6.427842,1155del32 D111 vs R270X pA D90,GO_Biological_Process_2025
3,Lytic Vacuole Membrane (GO:0098852),4.291520e-07,4.151190,1155del32 D111 vs R270X pA D90,GO_Cellular_Component_2025
4,Lysosomal Membrane (GO:0005765),1.391321e-06,3.595585,1155del32 D111 vs R270X pA D90,GO_Cellular_Component_2025
5,Cadherin Binding (GO:0045296),1.528400e-05,3.781220,1155del32 D111 vs R270X pA D90,GO_Molecular_Function_2025
6,Ubiquitin-Like Protein Ligase Binding (GO:0044389),2.132971e-02,2.867179,1155del32 D111 vs R270X pA D90,GO_Molecular_Function_2025


In [27]:
# Generate dot plot
if (nrow(all_data) > 0) {
  
  # Create database abbreviations and assign colors
  all_data <- all_data %>%
    mutate(
      DB_Abbrev = ifelse(Database %in% names(database_abbreviations), 
                         database_abbreviations[Database], Database),
      Module = factor(Module),
      # Create simple term label with database
      Simple_Term = paste(DB_Abbrev, Term, sep = ": ")
    )
  
  # Create a color mapping for databases
  unique_dbs <- unique(all_data$DB_Abbrev)
  db_colors <- setNames(RColorBrewer::brewer.pal(max(3, length(unique_dbs)), "Set3")[1:length(unique_dbs)], 
                       unique_dbs)
  
  # Sort terms by database first, then by term name
  all_data <- all_data %>%
    arrange(DB_Abbrev, Term)
  
  # Ensure terms are ordered properly
  term_order <- unique(all_data$Simple_Term)
  all_data$Simple_Term <- factor(all_data$Simple_Term, levels = term_order)
  
  dot_plot <- ggplot(all_data, aes(x = Module, y = Simple_Term, size = Odds.Ratio, fill = Adjusted.P.value)) +
    geom_point(shape = 21, alpha = 0.8) +
    scale_fill_viridis(name = 'Adjusted P-value', direction = -1) +
    scale_size_continuous(name = 'Odds Ratio', range = c(1, 8)) +
    xlab('') + ylab('') +
    labs(title = 'Significant Enrichment Terms Across Contrasts') +
    theme_minimal() +
    theme(
      panel.background = element_rect(fill = "white", color = NA),
      panel.grid.major = element_line(color = "grey90"),
      panel.grid.minor = element_blank(),
      axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5),
      axis.text.y = element_text(size = 10),
      legend.position = "right"
    )
  
  # Save the plot
  ggsave(filename = "06_DEG_GO/combined_top_terms_dot_plot.pdf", 
         plot = dot_plot, height = 15, width = 13)
}

In [26]:
# Save data
write.csv(all_data, "06_DEG_GO/combined_top_terms_dot_plot.csv", row.names = FALSE)